# 🎓 BÀI TOÁN DỰ ĐOÁN TỶ LỆ BỎ HỌC DỰA TRÊN CÁC CHỈ SỐ CĂNG THẲNG VÀ SỰ THAM GIA HỌC TẬP
## *(Student Burnout & Dropout Risk Prediction using Machine Learning)*

---
- **Chương trình:** Samsung Innovation Campus (SIC) - AI Capstone Project
- **Đề tài:** Xây dựng, huấn luyện và so sánh các mô hình học máy cho bài toán dự đoán tỷ lệ bỏ học dựa trên các chỉ số về căng thẳng và sự tham gia học tập (*Student Burnout*)
- **Thực hiện bởi Nhóm 03:**
  1. **Đỗ Quang Minh**
  2. **Vũ Quang Huy**

---
### 📌 Mục tiêu đồ án (Project Objectives)
1. **Khám phá và phân tích dữ liệu (EDA):** Khảo sát 800 hồ sơ sinh viên với 25 thuộc tính về nhân khẩu học, mức độ tham gia học tập (*Attendance, Study Hours, GPA, Motivation, Backlogs*), lối sống (*Sleep, Screen Time, Exercise*) và sức khỏe tâm lý (*Stress, Anxiety, Financial Stress, Peer Pressure, Family Support, Burnout Level*).
2. **Tiền xử lý & Kỹ thuật đặc trưng (Preprocessing & Feature Engineering):** Xử lý dữ liệu khuyết trên 18/25 cột bằng `KNNImputer` kết hợp `SimpleImputer`, mã hóa biến phân loại, và xây dựng các chỉ số tổng hợp phản ánh sự tương tác giữa **Căng thẳng (Stress)** và **Tham gia học tập (Academic Engagement)**.
3. **Huấn luyện & So sánh 10 Mô hình Học máy (Model Training & Comparison):** Đánh giá bằng kiểm định chéo phân tầng 5-Fold (`5-Fold Stratified Cross-Validation`) và tập kiểm thử độc lập (`Hold-out Test Set 20%`) trên 2 kịch bản:
   - **Kịch bản 1 (Full Feature Set):** Sử dụng toàn bộ đặc trưng bao gồm cả trạng thái kiệt sức (`Burnout_Level`).
   - **Kịch bản 2 (Early Warning - Direct Stress & Engagement):** Dự đoán nguy cơ bỏ học trực tiếp từ các chỉ số hành vi và tâm lý ban đầu khi chưa có đánh giá `Burnout_Level`.

## 1. Khai báo thư viện và Tải dữ liệu (Import Libraries & Load Dataset)

In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, VotingClassifier
)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Đọc dữ liệu
DATA_PATH = "data/student_burnout_dropout_dataset_2.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "student_burnout_dropout_dataset_2.csv"

df = pd.read_csv(DATA_PATH)
print(f"Kích thước tập dữ liệu (Shape): {df.shape}")
df.head()

Kích thước tập dữ liệu (Shape): (800, 25)

## 2. Khám phá Dữ liệu & Phân tích Dữ liệu Khuyết (Exploratory Data Analysis - EDA)
Tập dữ liệu gồm **800 sinh viên** và **25 cột**:
- **Định danh (1 cột):** `Student_ID`
- **Nhân khẩu học & Bối cảnh (6 cột):** `Age`, `Gender`, `Year_of_Study`, `Department`, `Residence_Type`, `Family_Income_Bracket`
- **Sự tham gia & Kết quả học tập (5 cột):** `Attendance_Percent`, `Study_Hours_Per_Day`, `Previous_GPA`, `Backlogs`, `Motivation_Score`
- **Lối sống & Sinh hoạt (5 cột):** `Sleep_Hours`, `Screen_Time_Hours`, `Exercise_Freq_Per_Week`, `Social_Activity_Score`, `Part_Time_Job`
- **Chỉ số Căng thẳng & Hỗ trợ tâm lý (6 cột):** `Financial_Stress_Score`, `Family_Support_Score`, `Stress_Level`, `Anxiety_Score`, `Peer_Pressure_Score`, `Counseling_Access`
- **Trạng thái Kiệt sức & Biến mục tiêu (2 cột):** `Burnout_Level` (`Low`, `Medium`, `High`) và `Dropout_Risk` (`Yes`, `No`).

In [1]:
# Kiểm tra tỷ lệ dữ liệu khuyết (Missing Values)
missing_df = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent (%)": (df.isnull().sum() / len(df) * 100).round(2),
    "Data_Type": df.dtypes.astype(str)
})
print("=== THỐNG KÊ CÁC CỘT CÓ DỮ LIỆU KHUYẾT (18/25 CỘT) ===")
print(missing_df[missing_df["Missing_Count"] > 0].sort_values("Missing_Count", ascending=False))

print("\n=== PHÂN PHỐI BIẾN MỤC TIÊU DROPOUT_RISK & BURNOUT_LEVEL ===")
print(pd.crosstab(df["Burnout_Level"], df["Dropout_Risk"], margins=True, normalize="index").round(4) * 100)

=== THỐNG KÊ CÁC CỘT CÓ DỮ LIỆU KHUYẾT (18/25 CỘT) ===
                        Missing_Count  Missing_Percent (%) Data_Type
Attendance_Percent                 51                 6.38   float64
Stress_Level                       49                 6.12   float64
Family_Support_Score               44                 5.50   float64
Financial_Stress_Score             43                 5.38   float64
Motivation_Score                   41                 5.12   float64
Peer_Pressure_Score                41                 5.12   float64
Social_Activity_Score              40                 5.00   float64
Anxiety_Score                      39                 4.88   float64
Screen_Time_Hours                  36                 4.50   float64
Study_Hours_Per_Day                35                 4.38   float64
Previous_GPA                       35                 4.38   float64
Sleep_Hours                        35                 4.38   float64
Gender                             30           

### 📊 Nhận xét quan trọng từ phân tích EDA:
1. **Cân bằng nhãn (Class Balance):** Biến mục tiêu `Dropout_Risk` có phân phối khá cân bằng: **437 sinh viên có nguy cơ bỏ học (`Yes` - 54.62%)** và **363 sinh viên không có nguy cơ (`No` - 45.38%)**.
2. **Tác động mạnh mẽ của `Burnout_Level`:**
   - Nhóm `Burnout_Level = High` có tỷ lệ bỏ học lên tới **92.00% (184/200)**.
   - Nhóm `Burnout_Level = Medium` có tỷ lệ bỏ học **63.21% (177/280)**.
   - Nhóm `Burnout_Level = Low` chỉ có tỷ lệ bỏ học **23.75% (76/320)**.
3. **Tương quan với Sự tham gia học tập & Căng thẳng:**
   - Các yếu tố **giảm nguy cơ bỏ học** mạnh nhất (tương quan âm): `Attendance_Percent` ($r = -0.244$), `Motivation_Score` ($r = -0.193$), `Study_Hours_Per_Day` ($r = -0.160$), `Family_Support_Score` ($r = -0.137$), `Previous_GPA` ($r = -0.106$).
   - Các yếu tố **tăng nguy cơ bỏ học** mạnh nhất (tương quan dương): `Burnout_Level` ($r = +0.550$), `Stress_Level` ($r = +0.164$), `Anxiety_Score` ($r = +0.145$), `Screen_Time_Hours` ($r = +0.127$), `Financial_Stress_Score` ($r = +0.114$).

## 3. Kỹ thuật Tạo Đặc trưng Chuyên sâu (Domain Feature Engineering) & Tiền xử lý
Để nâng cao chất lượng dự đoán của các mô hình học máy, nhóm thiết kế bộ biến đổi `DomainFeatureEngineer` nằm bên trong `Pipeline` của `scikit-learn` (đảm bảo **không rò rỉ dữ liệu - No Data Leakage** giữa tập huấn luyện và tập kiểm thử):
1. **`Academic_Engagement_Index`:** Chỉ số tổng hợp sự tham gia học tập từ chuyên cần, giờ tự học, động lực và GPA.
2. **`Psychological_Stress_Index`:** Chỉ số tổng hợp căng thẳng từ điểm Stress, Lo âu (Anxiety), Áp lực tài chính và Áp lực đồng trang lứa.
3. **`Stress_Engagement_Ratio`:** Tỷ số mất cân bằng giữa căng thẳng tâm lý và mức độ gắn kết học tập.
4. **`Net_Stress_Support_Balance`:** Hiệu số giữa áp lực tâm lý và điểm tựa hỗ trợ (Gia đình, Hoạt động xã hội, Thể chất).
5. **`Digital_Fatigue_Ratio`:** Tỷ lệ thời gian sử dụng màn hình (`Screen_Time_Hours`) trên thời gian ngủ (`Sleep_Hours`).

In [1]:
from src.train_and_compare_models import (
    DomainFeatureEngineer,
    build_preprocessor,
    get_models,
    evaluate_models_scenario,
)

# Chạy đánh giá toàn diện 10 mô hình trên 2 kịch bản
res_full, roc_full, pr_full, pipes_full = evaluate_models_scenario(
    df, include_burnout=True, scenario_name="Scenario 1: Full Features (With Burnout_Level)"
)
res_early, roc_early, pr_early, pipes_early = evaluate_models_scenario(
    df, include_burnout=False, scenario_name="Scenario 2: Early Warning (Without Burnout_Level)"
)

print("=== KỊCH BẢN 1: ĐẦY ĐỦ ĐẶC TRƯNG (CÓ BURNOUT_LEVEL & CHỈ SỐ TỔNG HỢP) ===")
display_cols = ["Model", "CV_Accuracy_Mean", "CV_F1_Mean", "CV_ROC_AUC_Mean", "Test_Accuracy", "Test_F1", "Test_ROC_AUC"]
print(res_full[display_cols].round(4).to_string(index=False))

print("\n=== KỊCH BẢN 2: CẢNH BÁO SỚM (CHỈ DÙNG STRESS & ENGAGEMENT THÔ) ===")
print(res_early[display_cols].round(4).to_string(index=False))

=== KỊCH BẢN 1: ĐẦY ĐỦ ĐẶC TRƯNG (CÓ BURNOUT_LEVEL & CHỈ SỐ TỔNG HỢP) ===
               Model  CV_Accuracy_Mean  CV_F1_Mean  CV_ROC_AUC_Mean  Test_Accuracy  Test_F1  Test_ROC_AUC
       Random Forest           0.74375    0.774586         0.805024        0.71875 0.751381      0.802393
Soft Voting Ensemble           0.73875    0.767936         0.799540        0.72500 0.758242      0.793733
 Logistic Regression           0.72250    0.751894         0.798186        0.73750 0.761364      0.798614
  MLP Neural Network           0.73625    0.759413         0.796750        0.70625 0.728324      0.777515
             XGBoost           0.71750    0.742787         0.792536        0.71250 0.738636      0.777673
         Extra Trees           0.75000    0.780976         0.792165        0.75625 0.789189      0.788065
   Gradient Boosting           0.71750    0.744415         0.790836        0.73125 0.751445      0.785231
    SVM (RBF Kernel)           0.74625    0.775652         0.785489        0.7

## 4. Trực quan hóa Kết quả So sánh các Mô hình Học máy
Toàn bộ các biểu đồ phân tích EDA và đánh giá mô hình đã được xuất với độ phân giải cao trong thư mục `figures/`:
- `figures/01_missing_values_analysis.png`
- `figures/02_target_and_burnout_distribution.png`
- `figures/03_stress_vs_engagement_distributions.png`
- `figures/04_correlation_matrix_heatmap.png`
- `figures/05_categorical_factors_analysis.png`
- `figures/06_multivariate_burnout_engagement_scatter.png`
- `figures/07_model_performance_comparison.png`
- `figures/08_roc_and_precision_recall_curves.png`
- `figures/09_confusion_matrices_top_models.png`
- `figures/10_feature_importance_analysis.png`

In [1]:
from IPython.display import Image, display

for fig_file in [
    "figures/02_target_and_burnout_distribution.png",
    "figures/07_model_performance_comparison.png",
    "figures/08_roc_and_precision_recall_curves.png",
    "figures/09_confusion_matrices_top_models.png",
    "figures/10_feature_importance_analysis.png",
]:
    if os.path.exists(fig_file):
        display(Image(filename=fig_file))